
MATCH_ID = 3869685  
TIME_ALVO = "Argentina"

In [66]:
# Importa as bibliotecas
from statsbombpy import sb
import pandas as pd
import numpy as np
import warnings
import networkx as nx

warnings.filterwarnings("ignore", category=UserWarning)

In [67]:
def analisar_varias_partidas_selecao(lista_match_ids, time_alvo, coluna_peso=None):
    """
    Recebe uma lista de match_ids de uma mesma seleção, executa a função
    calcular_pagerank_futebol para cada jogo e retorna a média de importância
    de cada jogador ao longo dessas partidas.
    """
    todos_rankings = []
    nome_coluna_pr = f'PageRank_{coluna_peso or "contagem"}'

    for match_id in lista_match_ids:
        print(f"\n--- Processando partida {match_id} ({time_alvo}) ---")
        ranking_jogo = calcular_pagerank_futebol(
            MATCH_ID=match_id,
            TIME_ALVO=time_alvo,
            coluna_peso=coluna_peso
        )
        ranking_jogo['match_id'] = match_id
        todos_rankings.append(ranking_jogo)

    # Junta os resultados de todos os jogos em um único DataFrame
    df_consolidado = pd.concat(todos_rankings, ignore_index=True)

    # Calcula a média de PageRank e a quantidade de jogos como titular de cada jogador
    df_media = df_consolidado.groupby('Jogador', as_index=False).agg(
        Jogos_Titular=('match_id', 'count'),
        Media_PageRank=(nome_coluna_pr, 'mean')
    )

    return df_media.sort_values(by='Media_PageRank', ascending=False).reset_index(drop=True)

In [68]:

def calcular_pagerank_futebol(MATCH_ID, TIME_ALVO, coluna_peso=None, vetor_personalizacao=None):
    
    # 1. Carrega o dataframe correspondente e calcula o alpha
    df_passes = obter_df_passes(MATCH_ID, TIME_ALVO)
    alpha = estimar_alpha_empirico(obter_eventos_titulares(MATCH_ID, TIME_ALVO))
    
    # 2. Agrega os passes entre cada par (passador -> recebedor)
    if coluna_peso is None:
        arestas = df_passes.groupby(['player', 'pass_recipient']).size().reset_index(name='peso_final')
    else:
        # Garante que não existam pesos negativos
        if (df_passes[coluna_peso] < 0).any():
            raise ValueError(f"A coluna '{coluna_peso}' contém valores negativos, o que viola a Cadeia de Markov.")
        arestas = df_passes.groupby(['player', 'pass_recipient'])[coluna_peso].sum().reset_index(name='peso_final')

    # 3. Constrói o grafo direcionado ponderado
    G = nx.DiGraph()
    for _, row in arestas.iterrows():
        G.add_edge(row['player'], row['pass_recipient'], weight=row['peso_final'])

    # 4. Executa o PageRank (algoritmo idêntico independente da métrica de peso)
    scores = nx.pagerank(
        G,
        alpha=alpha,
        weight='weight',
        personalization=vetor_personalizacao,
        tol=1e-08
    )

    # 5. Retorna ordenado em DataFrame para facilitar comparações
    ranking = pd.DataFrame(list(scores.items()), columns=['Jogador', f'PageRank_{coluna_peso or "contagem"}'])
    return ranking.sort_values(by=ranking.columns[1], ascending=False).reset_index(drop=True)

In [69]:

def obter_df_passes(MATCH_ID, TIME_ALVO):

    # 1. Baixa todos os eventos da partida
    eventos = sb.events(match_id=MATCH_ID)

    # Filtra apenas os eventos da equipe escolhida e ordena cronologicamente
    eventos_time = eventos[eventos['team'] == TIME_ALVO].sort_values(by=['period', 'minute', 'second', 'index']).copy()

    # 2. Identifica o momento exato da PRIMEIRA SUBSTITUIÇÃO do time
    subs = eventos_time[eventos_time['type'] == 'Substitution']
    if not subs.empty:
        primeira_sub_idx = subs.iloc[0]['index']
        minuto_sub = subs.iloc[0]['minute']
        eventos_titulares = eventos_time[eventos_time['index'] < primeira_sub_idx].copy()
        print(f"Recorte aplicado até a 1ª substituição aos {minuto_sub} min.")
    else:
        eventos_titulares = eventos_time.copy()
        print("Nenhuma substituição encontrada; partida completa utilizada.")

    # 3. Extrai e limpa os PASSES COMPLETOS
    passes = eventos_titulares[
        (eventos_titulares['type'] == 'Pass') & 
        (eventos_titulares['pass_outcome'].isna())  # NaN = passe completo
    ].dropna(subset=['player', 'pass_recipient', 'location', 'pass_end_location']).copy()
    
    
    
    # Desembrulha as coordenadas [x, y] (Campo StatsBomb: 120x80, gol adversário em x=120, y=40)
    passes[['x_inicio', 'y_inicio']] = pd.DataFrame(passes['location'].tolist(), index=passes.index)
    passes[['x_fim', 'y_fim']] = pd.DataFrame(passes['pass_end_location'].tolist(), index=passes.index)

    # 4. Engenharia de Atributos para os Pesos das Arestas (w_ij)
    # A) Ganho linear em direção ao gol (Delta X)
    passes['delta_x'] = passes['x_fim'] - passes['x_inicio'] 
    passes['avanco_positivo'] = passes['delta_x'].clip(lower=0)

    # B) Redução da distância euclidiana até o centro do gol adversário (120, 40)
    dist_inicio = np.hypot(120 - passes['x_inicio'], 40 - passes['y_inicio'])
    dist_fim = np.hypot(120 - passes['x_fim'], 40 - passes['y_fim'])
    passes['ganho_proximidade_gol'] = (dist_inicio - dist_fim).clip(lower=0)

    # C) Flags táticas (converte NaN para False/0)
    passes['sob_pressao'] = passes['under_pressure'].fillna(False).astype(int)
    passes['passe_para_chute'] = passes['pass_shot_assist'].fillna(False).astype(int)

    # D) Peso Composto Sugerido (exemplo prontamente utilizável no NetworkX):
    # Peso base 1.0 + ganho normalizado em direção ao gol + bônus por pressão (1.5x) ou assistência para chute (+5.0)
    passes['peso_tatico'] = (
        1.0 
        + (passes['ganho_proximidade_gol'] / 10.0) 
        + (passes['passe_para_chute'] * 5.0)
    ) * np.where(passes['sob_pressao'] == 1, 1.5, 1.0)

    CONSTANTE_BASE_XT = 0.01
    xt_inicio = obter_xt(passes['x_inicio'], passes['y_inicio'])
    xt_fim = obter_xt(passes['x_fim'], passes['y_fim'])
    passes['delta_xt'] = xt_fim - xt_inicio
    passes['peso_xt'] = CONSTANTE_BASE_XT + passes['delta_xt'].clip(lower=0)


    # 5. Dataset Principal de Arestas (Passes)
    colunas_passes = [
        'minute', 'second', 'player', 'pass_recipient',
        'x_inicio', 'y_inicio', 'x_fim', 'y_fim',
        'delta_x', 'avanco_positivo', 'ganho_proximidade_gol',
        'sob_pressao', 'passe_para_chute', 'peso_tatico', 'peso_xt'
    ]
    df_passes = passes[colunas_passes].reset_index(drop=True)

    # 6. Dataset Auxiliar: Inícios de Posse (para o vetor de teletransporte v) e Finalizações (nó absorvedor)
    recuperacoes = eventos_titulares[eventos_titulares['type'].isin(['Ball Recovery', 'Interception'])]
    vetor_inicio_posse = recuperacoes['player'].value_counts(normalize=True).to_dict()

    finalizacoes = eventos_titulares[eventos_titulares['type'] == 'Shot'][['minute', 'second', 'player', 'shot_outcome', 'shot_statsbomb_xg']]

    print(f"Passes completos válidos: {len(df_passes)}")
    print(f"Jogadores titulares no grafo: {df_passes['player'].nunique()}")
    return df_passes

In [70]:

def estimar_alpha_empirico(eventos_titulares):
    
    # 1. Passes que mantêm o fluxo dentro do grafo
    passes_completos = eventos_titulares[
        (eventos_titulares['type'] == 'Pass') & 
        (eventos_titulares['pass_outcome'].isna())
    ]
    n_continuacoes = len(passes_completos)

    # 2. Eventos que encerram a cadeia de passes daquela posse
    passes_errados = eventos_titulares[
        (eventos_titulares['type'] == 'Pass') & 
        (eventos_titulares['pass_outcome'].notna())
    ]
    chutes = eventos_titulares[eventos_titulares['type'] == 'Shot']
    perdas = eventos_titulares[eventos_titulares['type'].isin(['Dispossessed', 'Miscontrol'])]

    n_encerramentos = len(passes_errados) + len(chutes) + len(perdas)

    # Probabilidade empírica de a posse continuar por mais um passe
    alpha = n_continuacoes / (n_continuacoes + n_encerramentos)
    
    return alpha

In [71]:

def obter_eventos_titulares(match_id, time_alvo):

    eventos = sb.events(match_id=match_id)
    
    # Filtra o time e ordena cronologicamente
    eventos_time = eventos[eventos['team'] == time_alvo].sort_values(
        by=['period', 'minute', 'second', 'index']
    ).copy()

    # Localiza o índice da primeira substituição
    subs = eventos_time[eventos_time['type'] == 'Substitution']
    if not subs.empty:
        primeira_sub_idx = subs.iloc[0]['index']
        eventos_titulares = eventos_time[eventos_time['index'] < primeira_sub_idx].copy()
    else:
        eventos_titulares = eventos_time.copy()

    return eventos_titulares

In [72]:
# Matriz oficial 8x12 de Expected Threat (Karun Singh)
XT_GRID = np.array([
    [0.00638303, 0.00779616, 0.00844854, 0.00977659, 0.01126267, 0.01248344, 0.01473596, 0.01745060, 0.02122129, 0.02756312, 0.03485072, 0.03792590],
    [0.00750072, 0.00878589, 0.00942382, 0.01059490, 0.01214719, 0.01384540, 0.01611813, 0.01870347, 0.02401521, 0.02953272, 0.04066992, 0.04647721],
    [0.00887990, 0.00977745, 0.01001304, 0.01110462, 0.01269174, 0.01429128, 0.01685596, 0.01935132, 0.02412240, 0.02855202, 0.05491138, 0.06442595],
    [0.00941056, 0.01082722, 0.01016549, 0.01132376, 0.01262646, 0.01484598, 0.01689528, 0.01997070, 0.02385149, 0.03511326, 0.10805102, 0.25745362],
    [0.00941056, 0.01082722, 0.01016549, 0.01132376, 0.01262646, 0.01484598, 0.01689528, 0.01997070, 0.02385149, 0.03511326, 0.10805102, 0.25745362],
    [0.00887990, 0.00977745, 0.01001304, 0.01110462, 0.01269174, 0.01429128, 0.01685596, 0.01935132, 0.02412240, 0.02855202, 0.05491138, 0.06442595],
    [0.00750072, 0.00878589, 0.00942382, 0.01059490, 0.01214719, 0.01384540, 0.01611813, 0.01870347, 0.02401521, 0.02953272, 0.04066992, 0.04647721],
    [0.00638303, 0.00779616, 0.00844854, 0.00977659, 0.01126267, 0.01248344, 0.01473596, 0.01745060, 0.02122129, 0.02756312, 0.03485072, 0.03792590]
])

def obter_xt(x, y):
    """Converte coordenadas StatsBomb (0..120, 0..80) nos índices da grade 8x12."""
    col = np.clip((x / 120.0 * 12).astype(int), 0, 11)
    lin = np.clip((y / 80.0 * 8).astype(int), 0, 7)
    return XT_GRID[lin, col]



In [75]:
# Exemplo: Quartas, Semifinal e Final da Argentina na Copa 2022
ids_jogos = [3869321, 3869519, 3869685]

ranking_medio = analisar_varias_partidas_selecao(
    lista_match_ids=ids_jogos,
    time_alvo="Argentina",
    coluna_peso='peso_xt'  # Ou None para contagem simples
)

print(ranking_medio)


--- Processando partida 3869321 (Argentina) ---
Recorte aplicado até a 1ª substituição aos 65 min.
Passes completos válidos: 295
Jogadores titulares no grafo: 11

--- Processando partida 3869519 (Argentina) ---
Recorte aplicado até a 1ª substituição aos 61 min.
Passes completos válidos: 243
Jogadores titulares no grafo: 11

--- Processando partida 3869685 (Argentina) ---
Recorte aplicado até a 1ª substituição aos 63 min.
Passes completos válidos: 317
Jogadores titulares no grafo: 11
                            Jogador  Jogos_Titular  Media_PageRank
0   Ángel Fabián Di María Hernández              1        0.138737
1               Alexis Mac Allister              3        0.123971
2            Rodrigo Javier De Paul              3        0.123338
3                    Enzo Fernandez              3        0.112553
4    Lionel Andrés Messi Cuccittini              3        0.110716
5            Leandro Daniel Paredes              1        0.098999
6           Nicolás Hernán Otamendi       